### installing needed packages

In [1]:
%%capture
!pip install chromadb
!pip install llama-index
!pip install llama-index-vector-stores-chroma
!pip install llama-index-embeddings-huggingface

### importing needed libraries

In [1]:
import os
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer
import gdown
import zipfile
from tqdm import tqdm
import random
import shutil

import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.core import Settings, SimpleDirectoryReader , Document
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from huggingface_hub import InferenceClient
import re
import ollama
import datetime

import torch

C:\Users\Digi Max\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### <span style="color:red">(run below cell if you dont have the dataset in the same directory)</span>

In [5]:
gdown.download(id = '1BROb8jjYJc-P1voevE9OLcptuF7UMLK7', output='./LLM_app_dataset_v2.zip')

# Create the directory
os.makedirs('./LLM_app_dataset', exist_ok=True)

with zipfile.ZipFile('./LLM_app_dataset_v2.zip', 'r') as zip_ref:
    zip_ref.extractall('./LLM_app_dataset')

Downloading...
From: https://drive.google.com/uc?id=1BROb8jjYJc-P1voevE9OLcptuF7UMLK7
To: e:\uni\term 6\AutomationProject\LLM_app_dataset_v2.zip
100%|██████████| 729k/729k [00:00<00:00, 1.05MB/s]


### function to get answers from llama

In [39]:
def llama3(prompt, systemtext):
    response = ollama.chat(model='llama3', messages=[
                    {"role": "system", "content": systemtext},
                    {"role": "user", "content": prompt},
                ])

    return response['message']['content']

### simple code to test above function

In [3]:
systemtext= "you are a good assistant"
prompt = "What is the capital of France?"
response = llama3(prompt, systemtext)
print(response)

The capital of France is Paris.


### generating test dataset for further use in our tests (dont run this cell if the folder `test_dataset` is already generated)

In [ ]:
dest_dir_in = os.path.join("test_dataset", "in")
dest_dir_out = os.path.join("test_dataset", "out")

os.makedirs(dest_dir_in, exist_ok=True)
os.makedirs(dest_dir_out, exist_ok=True)

counter = 1

for folder_name in os.listdir('.'):
    if folder_name == "classifier_dataset":
        continue
    
    folder_path = os.path.join('.', folder_name)
    if os.path.isdir(folder_path):
        in_folder_path = os.path.join(folder_path, 'in')
        out_folder_path = os.path.join(folder_path, 'out')
        
        if os.path.exists(in_folder_path) and os.path.exists(out_folder_path):
            in_files = os.listdir(in_folder_path)
            selected_in_files = random.sample(in_files, len(in_files) // 4)
            
            for in_file in selected_in_files:
                match = re.search(r'in(\d+)', in_file)
                if match:
                    identifier = match.group(1)
                    
                    out_file_pattern = f'out{identifier}.'
                    for potential_out_file in os.listdir(out_folder_path):
                        if potential_out_file.startswith(out_file_pattern):
                            out_file = potential_out_file
                            break
                    
                    _, in_ext = os.path.splitext(in_file)
                    _, out_ext = os.path.splitext(out_file)
                    shutil.move(os.path.join(in_folder_path, in_file), os.path.join(dest_dir_in, f"in{counter}{in_ext}"))
                    shutil.move(os.path.join(out_folder_path, out_file), os.path.join(dest_dir_out, f"out{counter}{out_ext}"))
                    print(f'moved "{in_file}" to "{dest_dir_in}/in{counter}{in_ext}" and "{out_file}" to "{dest_dir_out}/out{counter}{out_ext}')
                    print
                    counter += 1



### generating classifier test dataset for further use in our tests (dont run this cell if the folder `classifier_test_dataset` is already generated)

In [ ]:
dest_dir_in = os.path.join("classifier_test_dataset", "in")
dest_dir_out = os.path.join("classifier_test_dataset", "out")

os.makedirs(dest_dir_in, exist_ok=True)
os.makedirs(dest_dir_out, exist_ok=True)

counter = 1

for folder_name in os.listdir('.'):
    if folder_name != "classifier_dataset":
        continue
    
    folder_path = os.path.join('.', folder_name)
    if os.path.isdir(folder_path):
        in_folder_path = os.path.join(folder_path, 'in')
        out_folder_path = os.path.join(folder_path, 'out')
        
        if os.path.exists(in_folder_path) and os.path.exists(out_folder_path):
            in_files = os.listdir(in_folder_path)
            selected_in_files = random.sample(in_files, len(in_files) // 4)
            
            for in_file in selected_in_files:
                match = re.search(r'in(\d+)', in_file)
                if match:
                    identifier = match.group(1)
                    
                    out_file_pattern = f'out{identifier}.'
                    for potential_out_file in os.listdir(out_folder_path):
                        if potential_out_file.startswith(out_file_pattern):
                            out_file = potential_out_file
                            break
                    
                    _, in_ext = os.path.splitext(in_file)
                    _, out_ext = os.path.splitext(out_file)
                    shutil.move(os.path.join(in_folder_path, in_file), os.path.join(dest_dir_in, f"in{counter}{in_ext}"))
                    shutil.move(os.path.join(out_folder_path, out_file), os.path.join(dest_dir_out, f"out{counter}{out_ext}"))
                    print(f'moved "{in_file}" to "{dest_dir_in}/in{counter}{in_ext}" and "{out_file}" to "{dest_dir_out}/out{counter}{out_ext}')
                    
                    counter += 1



### Retrival Embedding Extraction

In [4]:
Does_Extract_Embeddings = True #@param {type:"boolean"}
Does_Extract_Classifier_Embeddings = True #@param {type:"boolean"}

In [5]:
os.makedirs('./rag_vector', exist_ok=True)
os.makedirs('./classifier_rag_vector' , exist_ok= True)
if Does_Extract_Embeddings:
    base_dir = "./"

    all_documents = []

    for root, dirs, files in os.walk(base_dir):

        if 'test_dataset' in dirs:
            dirs.remove('test_dataset')
        if 'classifier_dataset' in dirs:
            dirs.remove('classifier_dataset')
        if os.path.basename(root) == "in":
            in_folder_path = os.path.join(root)

            documents = SimpleDirectoryReader(in_folder_path).load_data()

            all_documents.extend(documents)

    vector_path = './rag_vector'
    db = chromadb.PersistentClient(path=vector_path)
    chroma_collection = db.get_or_create_collection("index_cleaned")
    vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    Settings.chunk_size = 128
    Settings.chunk_overlap = 20
    Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

    index = VectorStoreIndex.from_documents(all_documents,
                                            storage_context=storage_context,
                                            show_progress=True)

if Does_Extract_Classifier_Embeddings:
    base_dir = "./"
    
    Classifierall_documents = []

    for root, dirs, files in os.walk(base_dir):
        if 'test_dataset' in dirs:
            dirs.remove('test_dataset')
        if 'completion' in dirs:
            dirs.remove('completion')
        if 'diag' in dirs:
            dirs.remove('diag')
        if 'extract' in dirs:
            dirs.remove('extract')
        if 'help' in dirs:
            dirs.remove('help')
        if 'list' in dirs:
            dirs.remove('list')
        if 'login' in dirs:
            dirs.remove('login')
        if 'network_new' in dirs:
            dirs.remove('network_new')
        if 'observe' in dirs:
            dirs.remove('observe')
        if 'rag_vector' in dirs:
            dirs.remove('rag_vector')
        if 'remove' in dirs:
            dirs.remove('remove')
        if 'test' in dirs:
            dirs.remove('test')
        if 'user_new' in dirs:
            dirs.remove('user_new')
        if os.path.basename(root) == "in":
            in_folder_path = os.path.join(root)

            documents = SimpleDirectoryReader(in_folder_path).load_data()

            Classifierall_documents.extend(documents)
        else:
            continue

    vector_path = './classifier_rag_vector'
    Classifierdb = chromadb.PersistentClient(path=vector_path)
    Classifierchroma_collection = Classifierdb.get_or_create_collection("index_cleaned")
    Classifiervector_store = ChromaVectorStore(chroma_collection=Classifierchroma_collection)
    Classifierstorage_context = StorageContext.from_defaults(vector_store=Classifiervector_store)
    Settings.chunk_size = 128
    Settings.chunk_overlap = 20
    Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

    Classifierindex = VectorStoreIndex.from_documents(Classifierall_documents,
                                            storage_context=Classifierstorage_context,
                                            show_progress=True)

Generating embeddings: 100%|██████████| 334/334 [00:15<00:00, 21.35it/s]


In [6]:
print(documents[10].text)

Create a 5G-SA user named 'JohnSmith' and use google-dns to check the readiness.

network specification:
    network-name: NetCentral
    access-name: WirelessHub
    core-name: centralNode5
    stack: 5g-sa
    dnn: "operator"
    network-mode: "IPv4"
    service-type: eMBB
    differentiator: 0x000023
    band: n77


In [8]:
print(len(all_documents))
print(len(Classifierall_documents))

746
332


## BAAI RAG Inference

## RAG Pipeline

In [9]:
vector_path = './rag_vector'
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
Settings.llm = None
Settings.chunk_size = 128
Settings.chunk_overlap = 20

db = chromadb.PersistentClient(path=vector_path)
chroma_collection = db.get_or_create_collection("index_cleaned")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_vector_store(vector_store,
                                            storage_context=storage_context)

LLM is explicitly disabled. Using MockLLM.


In [10]:
vector_path = './classifier_rag_vector'
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
Settings.llm = None
Settings.chunk_size = 128
Settings.chunk_overlap = 20

Classifierdb = chromadb.PersistentClient(path=vector_path)
Classifierchroma_collection = Classifierdb.get_or_create_collection("index_cleaned")
Classifiervector_store = ChromaVectorStore(chroma_collection=Classifierchroma_collection)
Classifierstorage_context = StorageContext.from_defaults(vector_store=Classifiervector_store)
Classsifierindex = VectorStoreIndex.from_vector_store(Classifiervector_store,
                                            storage_context=Classifierstorage_context)

LLM is explicitly disabled. Using MockLLM.


In [11]:
top_k = 8
retriever = VectorIndexRetriever(index=index,
                                  similarity_top_k=top_k)
query_engine = RetrieverQueryEngine(retriever=retriever)

ClassifierRetriever = VectorIndexRetriever(index=Classsifierindex,
                                  similarity_top_k=top_k)
Classifier_query_engine = RetrieverQueryEngine(retriever=ClassifierRetriever)

In [12]:
instruct = "Is it possible for you to examine the network graph for any bugs and make necessary corrections?" #@param {type:"string"}
retrivals = query_engine.query(instruct)
ClassifierRetrivals = Classifier_query_engine.query(instruct)
context = retrivals.source_nodes[0].text
prompt = f"""Instruction: With this context

  {context}

  {instruct}
  Output:
  """
systemtext = """Hi, You are an assistant to create a command based on the user input in the only specific valid format. The input is a message in which someone is trying to get the system to perform the diagnostics against your cluster. This command checks the following:

The cluster is reachable
The nodes in the cluster are healthy
The requested operators are installed successfully
The Athena operator is running and ready
If any networks are defined, whether they are ready
This command also tries to detect the issue and provide possible solutions.


The right format of the output is:
br-t9s.cli diag

"""
ans = llama3(prompt,systemtext)
print(prompt + ans)
print("-------------------------")



Instruction: With this context

  My goal is to check the network graph, but I'm uncertain about the procedure. Can you guide me through it?

  Is it possible for you to examine the network graph for any bugs and make necessary corrections?
  Output:
  br-t9s.cli diag
-------------------------


In [ ]:
print(len(retrivals.source_nodes))
for i in range(len(retrivals.source_nodes)):
  print(retrivals.source_nodes[i])

print(len(ClassifierRetrivals.source_nodes))
for i in range(len(ClassifierRetrivals.source_nodes)):
  print(ClassifierRetrivals.source_nodes[i])

In [18]:
def generatePrompt(prompt,retrivedDict):
  generatedPrompt = f"the input you have to give me output for is: {prompt}\n here are some similar inputs and outputs retrieved from the dataset:\n"
  for key in retrivedDict:
    generatedPrompt += f"input : {key} , output: {retrivedDict[key]}\n"
  
  generatedPrompt += f"give me the output for this input: {prompt}."
  return generatedPrompt


def generateRetrivedDict(prompt,retrival):
  retrivedDict = {}
  classifierRetrivedDict = {}
  base_dir = "./"
  for i in range(len(retrival.source_nodes)):
    if retrival.source_nodes[i].text == prompt :
      pass
    else:
      for root, dirs, files in os.walk(base_dir):
        if os.path.basename(root) == "in" :
          in_folder_path = os.path.join(root)
          for file_name in os.listdir(in_folder_path):
            if file_name.endswith('.txt'):
              in_file_path = os.path.join(in_folder_path, file_name)
              with open(in_file_path, 'r' , encoding='utf-8') as file:
                in_text = file.read()
              if in_text == retrival.source_nodes[i].text:
                fileOut = in_file_path.split("\\")
                if "classifier_dataset" not in fileOut[0] :
                  fileOut[-2] = "out"
                  fileOut[-1] = fileOut[-1].replace("in" , "out",1)
                  if fileOut[0] == "network" or fileOut[0] == "user":
                    fileOut[-1] = fileOut[-1].replace("txt" , "yaml",1)
                  outpath = '/'.join(fileOut)
                  with open(outpath, 'r') as file:
                    out_text = file.read()
                  retrivedDict.__setitem__(in_text , out_text)
                  classifierRetrivedDict.__setitem__(in_text , fileOut[0][2:])
                  break

  return retrivedDict , classifierRetrivedDict

def generateClassifierRetrivedDict(prompt,retrival):
  retrivedDict = {}
  base_dir = "./"
  for i in range(len(retrival.source_nodes)):
    if retrival.source_nodes[i].text == prompt :
      pass
    else:
      for root, dirs, files in os.walk(base_dir):
        if os.path.basename(root) == "in" :
          in_folder_path = os.path.join(root)
          for file_name in os.listdir(in_folder_path):
            if file_name.endswith('.txt'):
              in_file_path = os.path.join(in_folder_path, file_name)
              with open(in_file_path, 'r' , encoding='utf-8') as file:
                in_text = file.read()
              if in_text == retrival.source_nodes[i].text:
                fileOut = in_file_path.split("\\")
                if "classifier_dataset" in fileOut[0] :
                  fileOut[-2] = "out"
                  fileOut[-1] = fileOut[-1].replace("in" , "out",1)
                  if fileOut[2] == "network" or fileOut[2] == "user" or fileOut[2] == "user_new":
                    fileOut[-1] = fileOut[-1].replace("txt" , "yaml",1)
                  outpath = '/'.join(fileOut)
                  with open(outpath, 'r') as file:
                    out_text = file.read()
                  retrivedDict.__setitem__(in_text , out_text)
                  break

  return retrivedDict


In [19]:
retrivedDict , classifierDict1 = generateRetrivedDict(instruct , retrivals)
classifierDict  = generateClassifierRetrivedDict(instruct , ClassifierRetrivals)
print(retrivedDict)
print(classifierDict1)
print(classifierDict)

{"My goal is to check the network graph, but I'm uncertain about the procedure. Can you guide me through it?": 'br-t9s.cli extract graph --help', "I'm interested in checking the network graph, but I could use some guidance since I'm not sure how to proceed.": 'br-t9s.cli extract graph --help', 'Could you please give me access to the network graph?': 'br-t9s.cli extract graph', 'It would be helpful if you could debug and resolve any problems with the network graph.': 'br-t9s.cli extract graph --verbose', 'Can you provide me with the network graph?': 'br-t9s.cli extract graph', 'It would be really helpful if you could pass along the network graph to me.': 'br-t9s.cli extract graph', "I'm looking for network graph options. If possible, could you bring them for me?": 'br-t9s.cli extract graph --help', 'I would appreciate it if you could share the network graph with me.': 'br-t9s.cli extract graph'}
{"My goal is to check the network graph, but I'm uncertain about the procedure. Can you guid

In [ ]:
generatedPrompt = generatePrompt(instruct , retrivedDict)
print(generatedPrompt)
print("-"*50)
classifierGeneratedPrompt1 = generatePrompt(instruct , classifierDict1)
print(classifierGeneratedPrompt1)
print("-"*50)
classifierGeneratedPrompt = generatePrompt(instruct , classifierDict)
print(classifierGeneratedPrompt)

### extracting the system file required for the actual prompt

In [34]:
def extractSystemText(prompt , classifierDict):
    base_dir = "systemText"
    systemClassifier_path = os.path.join(base_dir, 'system_classifier.txt')
    with open(systemClassifier_path, 'r' , encoding='utf-8') as file:
        systemClassifier_text = file.read()
    
    prompt += "\ngive me the output in 1 word only."
    systemText = llama3(prompt, systemClassifier_text)
    possibleFiles = ["classifier" , "completion" , "diag" , "extract" , "help" , "list" , "login" , "network" , "observe" , "remove" , "test" , "user"]
    for name in possibleFiles:
        if name in systemText:
            systemText = name
            break
    
    if systemText not in possibleFiles:
        if len(classifierDict) > 0:
            first_key = next(iter(classifierDict))
            systemText = classifierDict[first_key]
    
    print(systemText)

    systemfile_name = f"system_{systemText}.txt"
    systemfile_path = os.path.join(base_dir, systemfile_name)
    try:
        with open(systemfile_path, 'r' , encoding='utf-8') as file:
            classifierFile = file.read()
    except:
        classifierFile = ""
    
    return classifierFile, systemText
  

In [ ]:
classifierFile1,_ = extractSystemText(classifierGeneratedPrompt1 , classifierDict)
print(classifierFile1)
print("-"*50)
classifierFile,_ = extractSystemText(classifierGeneratedPrompt , classifierDict)
print(classifierFile)

In [12]:
def calculate_simple_accuracy(out_dir , outgpt_dir , check = False):

    matching_files = 0
    out_files = [f for f in os.listdir(out_dir) if f.startswith('out') and f.endswith('.txt')]

    total_files = len(out_files)
    if check:
        for out_file in out_files:
            out_file_path = os.path.join(out_dir, out_file)
            out_file_number = out_file.replace('out', '').replace('.txt', '')
            outgpt_file_name = f"response_in{out_file_number}.txt"
            outgpt_file_path = os.path.join(outgpt_dir, outgpt_file_name)

            if os.path.exists(outgpt_file_path):
                out_text = read_file(out_file_path)
                outgpt_text = read_file(outgpt_file_path)

                # Check if contents are exactly equal
                if out_text in outgpt_text:
                    matching_files += 1
                #else:
                    #print(f"Mismatch between {out_file} and {outgpt_file_name}")
            else:
                total_files -= 1
                print(f"File {outgpt_file_name} does not exist in the outgpt directory")
    
    else:
        for out_file in out_files:
            out_file_path = os.path.join(out_dir, out_file)
            out_file_number = out_file.replace('out', '').replace('.txt', '')
            outgpt_file_name = f"response_in{out_file_number}.txt"
            outgpt_file_path = os.path.join(outgpt_dir, outgpt_file_name)

            if os.path.exists(outgpt_file_path):
                out_text = read_file(out_file_path)
                outgpt_text = read_file(outgpt_file_path)

                if out_text == outgpt_text:
                    matching_files += 1
                #else:
                    #print(f"Mismatch between {out_file} and {outgpt_file_name}")
            else:
                total_files -= 1
                print(f"File {outgpt_file_name} does not exist in the outgpt directory")

    accuracy = (matching_files / total_files) * 100
    print(f"Accuracy: {accuracy:.2f}%")

def read_file(file_path):
    with open(file_path, 'r') as file:
        return file.read()

In [ ]:
base_dir = 'classifier_test_dataset'
system_file_path = os.path.join(base_dir, 'system_classifier.txt')
in_dir = os.path.join(base_dir, 'in')
directory_name1 = "./classifier_test_dataset/outLLAMA1"
directory_name2 = "./classifier_test_dataset/outLLAMA2"
os.makedirs(directory_name1, exist_ok=True)
os.makedirs(directory_name2, exist_ok=True)
out_dir1 = os.path.join(base_dir, 'outLLAMA1')
out_dir2 = os.path.join(base_dir, 'outLLAMA2')

# Read the system_classifier.txt file
with open(system_file_path, 'r') as file:
    system_text = file.read()

for file_name in os.listdir(in_dir):
    if file_name.endswith('.txt'):
        in_file_path = os.path.join(in_dir, file_name)

        with open(in_file_path, 'r') as file:
            in_text = file.read()
        retrivals = Classifier_query_engine.query(in_text)
        ClassifierRetrivals = Classifier_query_engine.query(in_text)
        retrivedDict, classifierDict1 = generateRetrivedDict(in_text, retrivals)
        classifierDict2 = generateClassifierRetrivedDict(in_text, ClassifierRetrivals)
        classifierGeneratedPrompt1 = generatePrompt(in_text, classifierDict1)
        classifierGeneratedPrompt2 = generatePrompt(in_text, classifierDict2)
        print(in_text)
        _, systemText1 = extractSystemText(classifierGeneratedPrompt1, classifierDict1)
        _, systemText2 = extractSystemText(classifierGeneratedPrompt2, classifierDict2)
        print("-"*50)
        out_file_path1 = os.path.join(out_dir1, f"response_{file_name}")
        out_file_path2 = os.path.join(out_dir2, f"response_{file_name}")
        try:
            with open(out_file_path1, 'w') as file:
                file.write(systemText1)
            with open(out_file_path2, 'w') as file:
                file.write(systemText2)
        except Exception as e:
            print(f"Failed to process {file_name} due to error: {e}")


        

In [41]:
out_dir =  os.path.join(base_dir, 'out')
outgpt_dir1 = os.path.join(base_dir, 'outLLAMA1')
outgpt_dir2 = os.path.join(base_dir, 'outLLAMA2')

calculate_simple_accuracy(out_dir, outgpt_dir1)
calculate_simple_accuracy(out_dir, outgpt_dir2)

Mismatch between out14.txt and response_in14.txt
Mismatch between out30.txt and response_in30.txt
Mismatch between out34.txt and response_in34.txt
Mismatch between out44.txt and response_in44.txt
Accuracy: 94.29%
Mismatch between out12.txt and response_in12.txt
Mismatch between out14.txt and response_in14.txt
Mismatch between out26.txt and response_in26.txt
Mismatch between out43.txt and response_in43.txt
Mismatch between out44.txt and response_in44.txt
Mismatch between out54.txt and response_in54.txt
Accuracy: 91.43%


In [ ]:
base_dir = 'classifier_test_dataset'
system_file_path = os.path.join(base_dir, 'system_classifier.txt')
in_dir = os.path.join(base_dir, 'in')
directory_name = "./classifier_test_dataset/outLLAMA"
os.makedirs(directory_name, exist_ok=True)
out_dir = os.path.join(base_dir, 'outLLAMA')

# Read the system_classifier.txt file
with open(system_file_path, 'r') as file:
    system_text = file.read()

for file_name in os.listdir(in_dir):
    if file_name.endswith('.txt'):
        in_file_path = os.path.join(in_dir, file_name)

        # Read the content of the input file
        with open(in_file_path, 'r') as file:
            in_text = file.read()

        # Create the prompt
        prompt = f"{in_text}\n give me an only one word as the classification output."

        try:
            # Get the GPT-4 response
            response_text = llama3(prompt, system_text)

            # Save the response to a file in the 'out' directory
            out_file_path = os.path.join(out_dir, f"response_{file_name}")
            with open(out_file_path, 'w') as out_file:
                out_file.write(response_text)

            # Print the response (optional)
            print(f"Response for {file_name}:\n{response_text}\n")
        except Exception as e:
            print(f"Failed to process {file_name} due to error: {e}")

In [44]:
out_dir =  os.path.join(base_dir, 'out')
outgpt_dir = os.path.join(base_dir, 'outLLAMA')

calculate_simple_accuracy(out_dir, outgpt_dir)

Mismatch between out12.txt and response_in12.txt
Mismatch between out14.txt and response_in14.txt
Mismatch between out19.txt and response_in19.txt
Mismatch between out26.txt and response_in26.txt
Mismatch between out28.txt and response_in28.txt
Mismatch between out29.txt and response_in29.txt
Mismatch between out31.txt and response_in31.txt
Mismatch between out34.txt and response_in34.txt
Mismatch between out35.txt and response_in35.txt
Mismatch between out44.txt and response_in44.txt
Mismatch between out45.txt and response_in45.txt
Mismatch between out48.txt and response_in48.txt
Mismatch between out52.txt and response_in52.txt
Mismatch between out54.txt and response_in54.txt
Mismatch between out55.txt and response_in55.txt
Mismatch between out61.txt and response_in61.txt
Mismatch between out63.txt and response_in63.txt
Mismatch between out65.txt and response_in65.txt
Accuracy: 74.29%


as it is shown in above blocks the accurecy for these 3 approaches are in table 1
<div style="display: flex; justify-content: center;text-align: center;">

|               | Classifier Retriever | Base Retriever | Zero Shot |
|---------------|----------------------|----------------|-----------|
| LLaMA3        |94.29%                |91.43%          |74.29%     |
| Fine-tuned LLaMA3 |                  |                |           |

</div>


In [51]:
def extract_command(output_text):
    pattern = r'br-t9s\.cli[^\n]*'
    
    match = re.search(pattern, output_text)
    
    if match:
        command = match.group(0).strip('\'"')
        return command
    else:
        return None

In [ ]:
base_dir = 'test_dataset'
in_dir = os.path.join(base_dir, 'in')
directory_name1 = "./test_dataset/outLLAMA1"
directory_name2 = "./test_dataset/outLLAMA2"
os.makedirs(directory_name1, exist_ok=True)
os.makedirs(directory_name2, exist_ok=True)
out_dir1 = os.path.join(base_dir, 'outLLAMA1')
out_dir2 = os.path.join(base_dir, 'outLLAMA2')

for file_name in os.listdir(in_dir):
    if file_name.endswith('.txt'):
        in_file_path = os.path.join(in_dir, file_name)

        with open(in_file_path, 'r') as file:
            in_text = file.read()
        retrivals = Classifier_query_engine.query(in_text)
        ClassifierRetrivals = Classifier_query_engine.query(in_text)
        retrivedDict, classifierDict1 = generateRetrivedDict(in_text, retrivals)
        classifierDict2 = generateClassifierRetrivedDict(in_text, ClassifierRetrivals)
        classifierGeneratedPrompt1 = generatePrompt(in_text, classifierDict1)
        classifierGeneratedPrompt2 = generatePrompt(in_text, classifierDict2)
        print(in_text)
        systemFile1, systemText1 = extractSystemText(classifierGeneratedPrompt1, classifierDict1)
        systemFile2, systemText2 = extractSystemText(classifierGeneratedPrompt2, classifierDict2)
        generatedPrompt = generatePrompt(in_text, retrivedDict)
        print(generatedPrompt)
        out1 = llama3(generatedPrompt, systemFile1)
        out2 = llama3(generatedPrompt, systemFile2)
        if systemText1 != "network" and systemText1 != "user" and systemText1 != "remove":
            out1 = extract_command(out1)
            print(out1)
            out_file_path1 = os.path.join(out_dir1, f"response_{file_name}")
            try:
                with open(out_file_path1, 'w') as file:
                    file.write(out1)
            except Exception as e:
                print(f"Failed to process {file_name} due to error: {e}")
        if systemText2 != "network" and systemText2 != "user" and systemText2 != "remove":
            out2 = extract_command(out2)
            print(out2)
            out_file_path2 = os.path.join(out_dir2, f"response_{file_name}")
            try:
                with open(out_file_path2, 'w') as file:
                    file.write(out2)
            except Exception as e:
                print(f"Failed to process {file_name} due to error: {e}")
        print("-"*50)

In [ ]:
out_dir =  os.path.join(base_dir, 'out')
outgpt_dir1 = os.path.join(base_dir, 'outLLAMA1')
outgpt_dir2 = os.path.join(base_dir, 'outLLAMA2')

calculate_simple_accuracy(out_dir, outgpt_dir1)
calculate_simple_accuracy(out_dir, outgpt_dir2)

In [ ]:
base_dir = 'test_dataset'
in_dir = os.path.join(base_dir, 'in')
directory_name1 = "./test_dataset/outLLAMA"
os.makedirs(directory_name1, exist_ok=True)
out_dir1 = os.path.join(base_dir, 'outLLAMA')

for file_name in os.listdir(in_dir):
    if file_name.endswith('.txt'):
        in_file_path = os.path.join(in_dir, file_name)

        with open(in_file_path, 'r') as file:
            in_text = file.read()
        ClassifierRetrivals = Classifier_query_engine.query(in_text)
        retrivedDict, classifierDict1 = generateRetrivedDict(in_text, retrivals)
        classifierGeneratedPrompt1 = generatePrompt(in_text, classifierDict1)
        print(in_text)
        systemFile1, systemText1 = extractSystemText(classifierGeneratedPrompt1, classifierDict1)
        out1 = llama3(in_text, systemFile1)
        if systemText1 != "network" and systemText1 != "user":
            out1 = extract_command(out1)
            print(out1)
            out_file_path1 = os.path.join(out_dir1, f"response_{file_name}")
            try:
                with open(out_file_path1, 'w') as file:
                    file.write(out1)
            except Exception as e:
                print(f"Failed to process {file_name} due to error: {e}")
        print("-"*50)

In [ ]:
out_dir =  os.path.join(base_dir, 'out')
outgpt_dir = os.path.join(base_dir, 'outLLAMA')

calculate_simple_accuracy(out_dir, outgpt_dir)

In [7]:
import os

def tokenize_text(text):
    # Tokenize the text by splitting on whitespace and removing any extra punctuation
    return set(text.strip().split())

def calculate_miou(out_dir, outgpt_dir, check=False):
    total_iou = 0.0
    num_files = 0
    
    out_files = [f for f in os.listdir(out_dir) if f.startswith('out') and f.endswith('.txt')]

    for out_file in out_files:
        out_file_path = os.path.join(out_dir, out_file)
        out_file_number = out_file.replace('out', '').replace('.txt', '')
        outgpt_file_name = f"response_in{out_file_number}.txt"
        outgpt_file_path = os.path.join(outgpt_dir, outgpt_file_name)

        if os.path.exists(outgpt_file_path):
            out_text = read_file(out_file_path)
            outgpt_text = read_file(outgpt_file_path)
            
            # Tokenize both texts
            out_tokens = tokenize_text(out_text)
            outgpt_tokens = tokenize_text(outgpt_text)
            
            # Calculate intersection and union
            intersection = out_tokens.intersection(outgpt_tokens)
            union = out_tokens.union(outgpt_tokens)
            
            if len(union) > 0:
                iou = len(intersection) / len(union)
                total_iou += iou
                num_files += 1
            else:
                print(f"No valid tokens to compare in {out_file} and {outgpt_file_name}")

        #else:
            #print(f"File {outgpt_file_name} does not exist in the outgpt directory")

    if num_files > 0:
        miou = total_iou / num_files
        print(f"Mean IoU: {miou*100:.2f}%")
    else:
        print("No files were compared.")

# Example usage:
# calculate_miou('path/to/output_dir', 'path/to/real_output_dir')


In [9]:
base_dir = 'test_dataset'
out_dir =  os.path.join(base_dir, 'out')
outgpt_dir1 = os.path.join(base_dir, 'outLLAMA1')
outgpt_dir2 = os.path.join(base_dir, 'outLLAMA2')

calculate_miou(out_dir, outgpt_dir1)
calculate_miou(out_dir, outgpt_dir2)

Mean IoU: 65.12%
Mean IoU: 68.06%


In [10]:
out_dir =  os.path.join(base_dir, 'out')
outgpt_dir = os.path.join(base_dir, 'outLLAMA')

calculate_miou(out_dir, outgpt_dir)

Mean IoU: 47.36%


In [22]:
base_dir = 'GPTtest/classifier_dataset'
out_dir =  os.path.join(base_dir, 'out')
outgpt_dir = os.path.join(base_dir, 'outgpt')

calculate_simple_accuracy(out_dir, outgpt_dir)
calculate_miou(out_dir, outgpt_dir)
num_files1 = os.listdir(out_dir)
print(len(num_files1))

Accuracy: 80.00%
Mean IoU: 80.00%
280


In [23]:
base_dir = 'GPTtest/completion'
out_dir =  os.path.join(base_dir, 'out')
outgpt_dir = os.path.join(base_dir, 'outgpt')

calculate_simple_accuracy(out_dir, outgpt_dir)
calculate_miou(out_dir, outgpt_dir)
num_files2 = os.listdir(out_dir)
print(len(num_files2))

Accuracy: 100.00%
Mean IoU: 100.00%
52


In [24]:
base_dir = 'GPTtest/diag'
out_dir =  os.path.join(base_dir, 'out')
outgpt_dir = os.path.join(base_dir, 'outgpt')

calculate_simple_accuracy(out_dir, outgpt_dir)
calculate_miou(out_dir, outgpt_dir)
num_files3 = os.listdir(out_dir)
print(len(num_files3))


Accuracy: 96.92%
Mean IoU: 98.72%
65
